# 02 · Spatial Hybrid v1

Bestehende Physics-v3-Besitzphasen werden anhand räumlicher Ballnähe zeitlich ausgerichtet. D/X und die Phasenfolge bleiben erhalten. Eingabe ist die NPZ aus Notebook 01; Ausgabe sind vollständige H/A/D/X-CSV-Dateien für 199.319 Frames. Der Decoder arbeitet offline.

[← Notebook 01](../01_vorbereitung/01_vorbereitung.ipynb) · [Notebook 03 →](../03_videoaktionen/03_videoaktionen.ipynb)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

## 1. Vorbereitete Eingaben laden

`reference` und `status` stammen aus Physics v3, `proximity_margin_H` aus den Team-Ball-Distanzen. Der gespeicherte Hybrid wird erst nach der Berechnung zum Vergleich verwendet.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "artifacts/spatial_hybrid_v1/inputs/spatial_hybrid_inputs_v1.npz").is_file())
with np.load(PROJECT_DIR / "artifacts/spatial_hybrid_v1/inputs/spatial_hybrid_inputs_v1.npz", allow_pickle=False) as blob:
    reference, status = blob["reference"].copy(), blob["status"].copy()
    proximity_margin_H = np.asarray(blob["proximity_margin_H"], dtype=float)
    expected_hybrid, halves = blob["expected_hybrid"].copy(), blob["halves"].copy()
    N_FRAMES, FPS = int(blob["n_frames"]), int(blob["fps"])

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = PROJECT_DIR / "output/team_spatial_hybrid_v1"
DIAGNOSTICS, FIGURES, SUBMISSIONS = [OUTPUT / name / RUN_ID for name in ("diagnostics", "figures", "submissions")]
for folder in (DIAGNOSTICS, FIGURES, SUBMISSIONS):
    folder.mkdir(parents=True, exist_ok=True)
LABELS = ("H", "A", "D", "X")
COLORS = {"H": "#3068A0", "A": "#CE515B", "D": "#A4A9AF", "X": "#F3E7B3"}

def show_figure(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

print("Frames:", N_FRAMES, "| FPS:", FPS)
print("Frames mit räumlicher Evidenz:", int(np.isfinite(proximity_margin_H).sum()))
display(pd.Series(reference).value_counts().reindex(LABELS, fill_value=0).to_frame("Physics v3"))

## 2. Räumliche Kandidaten und Decoder

Die Differenz `d_A − d_H` wird mit einem zentrierten Median über 11 Frames geglättet, mindestens 8 Werte müssen vorhanden sein. Direkte Vorzeichenwechsel liefern richtungsgebundene Kandidaten; Nullwerte oder fehlende geglättete Werte unterbrechen die Folge.

Für jeden bestehenden H/A-Wechsel wird der nächste passende Kandidat innerhalb höchstens ±3 s gewählt. Der Suchradius wird zusätzlich auf die halben Abstände zu benachbarten Wechseln begrenzt. Nähe allein erzeugt keine neue Besitzphase.

In [ ]:
def detect_spatial_crossings(margin):
    smooth = pd.Series(np.asarray(margin, dtype=float)).rolling(
        11, center=True, min_periods=8
    ).median().to_numpy()
    sign = np.zeros(len(smooth), dtype=np.int8)
    sign[smooth > 0] = 1
    sign[smooth < 0] = -1
    crossings = {}
    for side, target in (("H", 1), ("A", -1)):
        crossings[side] = np.flatnonzero(
            (sign[1:] == target) & (sign[:-1] == -target)
        ) + 1
    return smooth, sign, crossings

def live_blocks(status_values, half_bounds):
    active = np.zeros(len(status_values), dtype=bool)
    for start, end in half_bounds:
        active[int(start):int(end) + 1] = True
    live = active & (status_values == "L")
    edges = np.flatnonzero(np.diff(np.r_[False, live, False].astype(np.int8)))
    return live, edges[::2], edges[1::2]

def spatial_hybrid_v1(reference_values, status_values, crossing_frames, half_bounds, fps=25):
    """Verschiebt bestehende H/A-Grenzen zu passenden räumlichen Kandidaten."""
    live, starts, stops = live_blocks(status_values, half_bounds)
    output = np.asarray(reference_values).copy()
    radius_max = round(3.0 * fps)
    records = []
    original_switch_count = 0

    for seg_start, seg_stop in zip(starts, stops):
        if seg_stop - seg_start < 2:
            continue
        f = np.arange(seg_start + 1, seg_stop)
        left_labels = reference_values[f - 1]
        right_labels = reference_values[f]
        is_switch = (
            np.isin(left_labels, ["H", "A"])
            & np.isin(right_labels, ["H", "A"])
            & (left_labels != right_labels)
        )
        switches = f[is_switch]
        original_switch_count += len(switches)
        for i, frame in enumerate(switches):
            frame = int(frame)
            old_side, new_side = str(reference_values[frame-1]), str(reference_values[frame])
            left = int(switches[i-1]) if i else int(seg_start)
            right = int(switches[i+1]) if i+1 < len(switches) else int(seg_stop)
            radius = min(radius_max, (frame - left - 1)//2, (right - frame - 1)//2)
            if radius <= 0:
                continue
            possible = crossing_frames[new_side]
            lo = np.searchsorted(possible, frame - radius, side="left")
            hi = np.searchsorted(possible, frame + radius, side="right")
            candidates = possible[lo:hi]
            if not len(candidates):
                continue
            spatial_frame = int(candidates[np.argmin(np.abs(candidates - frame))])
            shift = spatial_frame - frame
            if shift == 0:
                continue
            if shift < 0:
                output[spatial_frame:frame] = new_side
            else:
                output[frame:spatial_frame] = old_side
            records.append({
                "original_frame": frame, "aligned_frame": spatial_frame,
                "shift_seconds": shift/fps, "from": old_side, "to": new_side,
                "segment_start": int(seg_start), "segment_end": int(seg_stop-1)
            })
    return output, pd.DataFrame(records, columns=[
        "original_frame", "aligned_frame", "shift_seconds", "from", "to",
        "segment_start", "segment_end"
    ]), original_switch_count, (live, starts, stops), crossing_frames

## 3. Hybrid über das gesamte Spiel berechnen

Der Decoder bearbeitet nur LIVE innerhalb der beiden Halbzeiten. Die Tabelle zeigt die verschobenen Grenzen und ihren Zeitversatz.

In [ ]:
margin_smooth, spatial_sign, spatial_crossings = detect_spatial_crossings(proximity_margin_H)
hybrid, df_alignments, n_original_switches, (live_mask, starts, stops), _ = spatial_hybrid_v1(
    reference, status, spatial_crossings, halves, fps=FPS
)
changed = hybrid != reference

print("=== GESAMTES SPIEL ===")
print("Direkte H/A-Wechsel:", n_original_switches)
print("Räumlich verschobene Wechsel:", len(df_alignments))
print("Veränderte Frames:", int(changed.sum()))
print("Räumliche Kandidaten H/A:", {k: len(v) for k,v in spatial_crossings.items()})
print("Frames mit Distanzdifferenz:", int(np.isfinite(proximity_margin_H).sum()))
print("Verschiebungsrichtung: früher",
      int((df_alignments["shift_seconds"] < 0).sum()), ", später",
      int((df_alignments["shift_seconds"] > 0).sum()))
display(df_alignments.head(12).round(3))

print("Abweichende Frames zur bisherigen v1-Vorhersage:", int(np.count_nonzero(hybrid != expected_hybrid)))

## 4. Wirkung der zeitlichen Verschiebung

Labelverteilung, Richtung und Größe der Verschiebungen zeigen den Umfang der Änderung. Das räumliche Beispiel wird über den größten Versatz ausgewählt.

In [ ]:
counts = pd.DataFrame({name: pd.Series(seq).value_counts().reindex(LABELS, fill_value=0)
                       for name, seq in (("Physics v3", reference), ("Spatial Hybrid v1", hybrid))})
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
counts.plot.bar(ax=ax, color=["#A4A9AF", "#3068A0"], rot=0)
ax.set(title="Labelverteilung · alle 199.319 Frames", xlabel="Label", ylabel="Frames")
show_figure(fig, "02_labelverteilung")
display(Markdown("**Beobachtung:** Die D-/X-Framezahlen bleiben exakt gleich; nur H/A-Grenzen werden verschoben."))

shift_counts = pd.crosstab(df_alignments["from"] + " → " + df_alignments["to"],
                          np.where(df_alignments["shift_seconds"] < 0, "Früher", "Später"))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
shift_counts.plot.bar(ax=axes[0], rot=0, color=["#3068A0", "#CE515B"])
axes[0].set(title="Richtung der verschobenen Grenzen", xlabel="Teamwechsel", ylabel="Anzahl")
axes[0].legend(title="Zeitpunkt")
axes[1].hist(df_alignments["shift_seconds"], bins=np.arange(-3.125, 3.126, 0.25), color=COLORS["H"])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set(title="Zeitliche Verschiebungen", xlabel="Neue − alte Grenze (s)", ylabel="Anzahl")
show_figure(fig, "03_verschiebungsrichtung_und_histogramm")
display(Markdown(f"**Beobachtung:** {int(df_alignments['shift_seconds'].lt(0).sum())} Grenzen wandern nach vorn, "
                 f"{int(df_alignments['shift_seconds'].gt(0).sum())} nach hinten. Der Median des Betrags beträgt "
                 f"{df_alignments['shift_seconds'].abs().median():.2f} s."))

example = df_alignments.loc[df_alignments["shift_seconds"].abs().idxmax()]
center = int(example["original_frame"])
lo, hi = max(0, center - 125), min(N_FRAMES, center + 126)
fig, ax = plt.subplots(figsize=(12, 3.3), layout="constrained")
ax.plot(np.arange(lo, hi) / FPS, margin_smooth[lo:hi], label="Median d_A − d_H")
ax.axhline(0, color="grey", linestyle="--")
ax.axvline(example["original_frame"] / FPS, color=COLORS["A"], linestyle="--", label="Physics v3")
ax.axvline(example["aligned_frame"] / FPS, color=COLORS["H"], label="Spatial Hybrid v1")
ax.set(title=f"Größte Verschiebung · Frame {center} · Auswahl ohne GT", xlabel="Taktische Videosekunde", ylabel="Distanzdifferenz (m)")
ax.legend()
show_figure(fig, "04_raeumliche_evidenz_beispiel")

## 5. Vorhersagen und Diagnosen speichern

Beide CSVs enthalten die Spalten `frame,label` und alle Frames ab 0. Die Diagnosetabelle hält die zeitlichen Verschiebungen fest.

In [ ]:
models = {"physics_v3_reference": reference, "spatial_hybrid_v1": hybrid}
submission_files = {}
for name, labels in models.items():
    file = SUBMISSIONS / f"{name}.csv"
    pd.DataFrame({"frame": np.arange(N_FRAMES), "label": labels}).to_csv(file, index=False)
    submission_files[name] = file
    print(name, "→", file)
df_alignments.to_csv(DIAGNOSTICS / "spatial_alignment_diagnostics.csv", index=False)

## 6. Historischer öffentlicher Vergleich

Die folgenden Werte stammen aus der bereits dokumentierten offiziellen Evaluation vom 28. Oktober; sie werden hier nicht neu berechnet. Das wiederholt verwendete öffentliche Entwicklungsfenster umfasst 7.500 Frames und ist kein unabhängiger Test.

In [ ]:
scores = pd.DataFrame([
    {"Modell": "Physics v3", "Spell-F1": 0.543590, "Macro-F1": 0.854311},
    {"Modell": "Spatial Hybrid v1", "Spell-F1": 0.564103, "Macro-F1": 0.865360},
])
display(scores)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for ax, metric in zip(axes, ("Spell-F1", "Macro-F1")):
    bars = ax.bar(scores.Modell, scores[metric], color=["#A4A9AF", COLORS["H"]])
    ax.bar_label(bars, fmt="%.4f", padding=4)
    ax.set(title=metric, ylabel="Historischer öffentlicher Score", ylim=(0, 1))
show_figure(fig, "05_historische_scores")

## Ergebnis

Im bisherigen Stand wurden 150 von 258 direkten H/A-Grenzen verschoben und insgesamt 2.143 Frames geändert. Der öffentliche Spell-F1 liegt historisch bei 0,5641. Übertragbarkeit auf andere Spiele ist nicht unabhängig belegt. Die folgenden Notebooks 03–05 untersuchen Videoaktionen als zusätzliche Evidenz.